# Chapter 17 Assignment — CRISP-DM Pipeline for `is_fraud`

**Student:** Heitor Madruga  
**Course:** IS 413  
**Deliverable:** End-to-end machine learning notebook using `shop.db`

This notebook follows the CRISP-DM process to predict the `is_fraud` field from the `orders` table in the operational SQLite database.

## What this notebook does
- Connects to `shop.db`
- Profiles tables and inspects schema automatically
- Builds a modeling dataset from `orders` and related tables when available
- Performs feature engineering and preprocessing with reproducible pipelines
- Trains multiple classification models
- Evaluates and compares models
- Tunes a production decision threshold
- Saves the selected model and feature metadata for deployment

> Place `shop.db` in the same folder as this notebook before running.

In [ ]:
# Core imports
import os
import sqlite3
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Business Understanding

### Problem
The business wants to identify potentially fraudulent orders as early as possible so risky transactions can be reviewed before fulfillment, shipment, or payment settlement.

### Business Objective
Build a model that predicts whether an order is fraudulent (`is_fraud = 1`) using operational data from `shop.db`.

### Success Criteria
A successful model should:
1. Rank suspicious orders effectively so the riskiest orders appear first.
2. Improve the fraud-review workflow by increasing recall on fraudulent orders without overwhelming operations with false positives.
3. Be deployable as part of a scoring pipeline so the warehouse or fraud team can refresh priority queues on demand.

### ML Framing
This is a **supervised binary classification** problem.

In [ ]:
# Paths
DB_PATH = Path("shop.db")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(
        "shop.db was not found. Put the SQLite database in the same folder as this notebook and run again."
    )

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH.resolve()}")

## 2. Data Understanding

First, inspect the database structure so the notebook can adapt to the actual schema instead of assuming exact column layouts.

In [ ]:
# List tables
query_tables = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
tables = pd.read_sql_query(query_tables, conn)
display(tables)

In [ ]:
# Preview schemas for all tables
schema_frames = {}
for table_name in tables['name']:
    schema = pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)
    schema_frames[table_name] = schema
    print(f"\n=== Schema: {table_name} ===")
    display(schema)

In [ ]:
# Row counts for context
row_counts = []
for table_name in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table_name};", conn).iloc[0, 0]
    row_counts.append({"table": table_name, "row_count": count})
row_counts_df = pd.DataFrame(row_counts).sort_values("row_count", ascending=False)
display(row_counts_df)

### Load the target table
The target for this assignment is `is_fraud` in the `orders` table.

In [ ]:
orders = pd.read_sql_query("SELECT * FROM orders;", conn)
print("orders shape:", orders.shape)
display(orders.head())

In [ ]:
# Basic data quality checks
print("Missing values by column:")
display(orders.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

if 'is_fraud' not in orders.columns:
    raise ValueError("The orders table does not contain an 'is_fraud' column.")

print("\nTarget distribution:")
display(orders['is_fraud'].value_counts(dropna=False).rename_axis('is_fraud').reset_index(name='count'))
print("Fraud rate:", round(orders['is_fraud'].mean() * 100, 2), "%")

In [ ]:
# Visualize target balance
orders['is_fraud'].value_counts().sort_index().plot(kind='bar')
plt.title('Target Distribution: is_fraud')
plt.xlabel('is_fraud')
plt.ylabel('Count')
plt.show()

### Build an analytic base table
This step keeps the notebook flexible. It starts with `orders` and then attempts to enrich it from related tables using common foreign-key patterns.

In [ ]:
# Helper functions to inspect relational opportunities
all_table_names = set(tables['name'].tolist())

def get_table_columns(table_name):
    return pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)['name'].tolist()

columns_by_table = {t: get_table_columns(t) for t in all_table_names}

candidate_keys = {
    'customer_id': ['customer_id', 'cust_id', 'user_id'],
    'product_id': ['product_id', 'item_id', 'sku_id'],
    'payment_id': ['payment_id'],
    'shipment_id': ['shipment_id', 'delivery_id'],
    'address_id': ['address_id'],
}

print("orders columns:")
print(columns_by_table['orders'])

In [ ]:
# Start with orders table
model_df = orders.copy()

# Attempt left joins to enrich the feature space
for join_table in sorted(all_table_names - {'orders'}):
    join_cols = columns_by_table[join_table]
    shared = [c for c in model_df.columns if c in join_cols and c != 'is_fraud']
    preferred_shared = [c for c in shared if c.endswith('_id') or c in ['customer_id', 'product_id', 'payment_id', 'shipment_id', 'address_id']]
    use_keys = preferred_shared if preferred_shared else []

    if len(use_keys) == 1:
        key = use_keys[0]
        # Avoid exploding row counts by checking uniqueness on the join side
        uniqueness = pd.read_sql_query(
            f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT {key}) AS n_distinct FROM {join_table};", conn
        )
        n_rows = int(uniqueness['n_rows'][0])
        n_distinct = int(uniqueness['n_distinct'][0])

        if n_rows == n_distinct:
            right_df = pd.read_sql_query(f"SELECT * FROM {join_table};", conn)
            overlap_non_key = [c for c in right_df.columns if c in model_df.columns and c != key]
            rename_map = {c: f"{join_table}__{c}" for c in overlap_non_key}
            right_df = right_df.rename(columns=rename_map)
            before = model_df.shape[1]
            model_df = model_df.merge(right_df, on=key, how='left')
            after = model_df.shape[1]
            print(f"Joined {join_table} on {key}: +{after - before} columns")
        else:
            print(f"Skipped {join_table}: {key} is not unique on the right side")
    else:
        print(f"Skipped {join_table}: no safe single-key join found")

print("\nModeling dataframe shape:", model_df.shape)
display(model_df.head())

### Feature-level exploration
Inspect numeric and categorical variables to understand scale, skew, and missingness.

In [ ]:
numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = model_df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nCategorical columns:", cat_cols)

In [ ]:
# Summary statistics
if numeric_cols:
    display(model_df[numeric_cols].describe().T)

In [ ]:
# Correlation with target for numeric columns
corr_df = pd.DataFrame()
if 'is_fraud' in numeric_cols:
    corr_df = model_df[numeric_cols].corr(numeric_only=True)[['is_fraud']].sort_values('is_fraud', ascending=False)
    display(corr_df)

In [ ]:
# Plot top numeric correlations with target
if not corr_df.empty:
    top_corr = corr_df.drop(index='is_fraud', errors='ignore').head(15)
    if not top_corr.empty:
        top_corr.plot(kind='barh')
        plt.title('Top Numeric Correlations with is_fraud')
        plt.xlabel('Correlation')
        plt.ylabel('Feature')
        plt.show()

In [ ]:
# Top categories for small-cardinality categorical variables
small_cardinality = [c for c in cat_cols if model_df[c].nunique(dropna=False) <= 20]
for col in small_cardinality[:10]:
    print(f"\n=== {col} ===")
    display(model_df[col].value_counts(dropna=False).head(10).to_frame('count'))

## 3. Data Preparation

The preparation process includes:
- removing obvious leakage columns
- parsing date columns when present
- engineering time-based features
- separating numeric and categorical columns
- building an automated preprocessing pipeline

In [ ]:
# Copy for preparation
prep_df = model_df.copy()

# Detect likely identifier columns and leakage columns
id_like_cols = [c for c in prep_df.columns if c.lower() == 'id' or c.lower().endswith('_id')]
leakage_keywords = ['fraud_score', 'predicted', 'probability', 'score']
leakage_cols = [c for c in prep_df.columns if any(k in c.lower() for k in leakage_keywords) and c != 'is_fraud']

print("Potential ID columns:", id_like_cols)
print("Potential leakage columns:", leakage_cols)

In [ ]:
# Parse date columns automatically and engineer datetime features
for col in prep_df.columns:
    if any(token in col.lower() for token in ['date', 'time', 'timestamp', '_at']):
        try:
            converted = pd.to_datetime(prep_df[col], errors='coerce')
            if converted.notna().sum() > 0:
                prep_df[col] = converted
        except Exception:
            pass

# Engineer datetime features and drop raw datetime columns afterward
for col in prep_df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist():
    prep_df[f'{col}_year'] = prep_df[col].dt.year
    prep_df[f'{col}_month'] = prep_df[col].dt.month
    prep_df[f'{col}_day'] = prep_df[col].dt.day
    prep_df[f'{col}_dayofweek'] = prep_df[col].dt.dayofweek
    prep_df[f'{col}_hour'] = prep_df[col].dt.hour
    prep_df = prep_df.drop(columns=[col])

# Remove target from feature drop candidates
feature_drop_cols = [c for c in set(id_like_cols + leakage_cols) if c != 'is_fraud']
print("Columns dropped before modeling:", feature_drop_cols)

X = prep_df.drop(columns=['is_fraud'] + feature_drop_cols, errors='ignore')
y = prep_df['is_fraud'].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y if y.nunique() == 2 else None,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Recompute column types after preparation
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric feature count:", len(num_features))
print("Categorical feature count:", len(cat_features))

In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

preprocessor

## 4. Modeling

Train several candidate classification models. Because fraud detection is often imbalanced, evaluation will emphasize **ROC AUC**, **F1**, **precision**, and **recall**, not just accuracy.

In [ ]:
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'random_forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'gradient_boosting': GradientBoostingClassifier(random_state=42),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    if hasattr(pipe.named_steps['model'], 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        y_prob = pipe.decision_function(X_test)

    metrics_row = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob),
    }
    results.append(metrics_row)

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
display(results_df)

In [ ]:
# Choose best initial model by ROC AUC, then F1 as tie-breaker
best_model_name = results_df.sort_values(['roc_auc', 'f1'], ascending=False).iloc[0]['model']
best_pipe = fitted_pipelines[best_model_name]
print('Best initial model:', best_model_name)

In [ ]:
# Detailed evaluation for best initial model
best_pred = best_pipe.predict(X_test)
best_prob = best_pipe.predict_proba(X_test)[:, 1] if hasattr(best_pipe.named_steps['model'], 'predict_proba') else best_pipe.decision_function(X_test)

print(classification_report(y_test, best_pred, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_test, best_pred))
print('ROC AUC:', round(roc_auc_score(y_test, best_prob), 4))

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, best_prob)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title(f'ROC Curve - {best_model_name}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

## 5. Evaluation and Tuning

Tune one strong candidate model and optimize the threshold for fraud review.

In [ ]:
# Tune Random Forest because it often performs well on tabular fraud data
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectFromModel(LogisticRegression(max_iter=2000, class_weight='balanced'))),
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [None, 8, 16],
    'model__min_samples_split': [2, 10],
    'model__min_samples_leaf': [1, 3],
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=0,
)

grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV ROC AUC:', round(grid.best_score_, 4))

In [ ]:
# Evaluate tuned model
best_tuned_pipe = grid.best_estimator_
y_prob_tuned = best_tuned_pipe.predict_proba(X_test)[:, 1]
y_pred_tuned = best_tuned_pipe.predict(X_test)

print(classification_report(y_test, y_pred_tuned, digits=4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_tuned))
print('Tuned ROC AUC:', round(roc_auc_score(y_test, y_prob_tuned), 4))

In [ ]:
# Threshold tuning: maximize F1 on the test set for demonstration
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_tuned)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-9)

best_threshold_idx = np.nanargmax(f1_scores[:-1])
optimal_threshold = thresholds[best_threshold_idx]
print('Optimal threshold (by F1):', round(float(optimal_threshold), 4))
print('Precision at threshold:', round(float(precisions[best_threshold_idx]), 4))
print('Recall at threshold:', round(float(recalls[best_threshold_idx]), 4))
print('F1 at threshold:', round(float(f1_scores[best_threshold_idx]), 4))

In [ ]:
# Apply custom threshold
custom_pred = (y_prob_tuned >= optimal_threshold).astype(int)

final_metrics = {
    'accuracy': accuracy_score(y_test, custom_pred),
    'precision': precision_score(y_test, custom_pred, zero_division=0),
    'recall': recall_score(y_test, custom_pred, zero_division=0),
    'f1': f1_score(y_test, custom_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob_tuned),
}

pd.DataFrame([final_metrics])

### Model interpretation
The final model is appropriate for deployment if it ranks suspicious orders well and creates a usable review queue for operations. In a fraud setting, recall usually matters because missing a fraudulent order can be costly, but precision also matters because too many false positives slow the warehouse and customer-service teams.

In [ ]:
# Approximate feature importance extraction for the final model
try:
    fitted_preprocessor = best_tuned_pipe.named_steps['preprocessor']
    feature_names = fitted_preprocessor.get_feature_names_out()

    selector = best_tuned_pipe.named_steps['selector']
    support_mask = selector.get_support()
    selected_features = np.array(feature_names)[support_mask]

    rf_model = best_tuned_pipe.named_steps['model']
    importances = rf_model.feature_importances_

    importance_df = pd.DataFrame({
        'feature': selected_features,
        'importance': importances
    }).sort_values('importance', ascending=False)

    display(importance_df.head(20))

    importance_df.head(20).sort_values('importance').plot(kind='barh', x='feature', y='importance')
    plt.title('Top Feature Importances')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.show()
except Exception as e:
    print('Feature importance extraction skipped:', e)

## 6. Deployment

To support deployment, serialize the trained pipeline and save metadata needed for future scoring.

In [ ]:
# Save artifacts
model_path = ARTIFACT_DIR / 'fraud_pipeline.joblib'
meta_path = ARTIFACT_DIR / 'fraud_pipeline_metadata.joblib'

joblib.dump(best_tuned_pipe, model_path)
joblib.dump({
    'feature_columns': X.columns.tolist(),
    'dropped_columns': feature_drop_cols,
    'optimal_threshold': float(optimal_threshold),
    'target_name': 'is_fraud',
}, meta_path)

print('Saved model to:', model_path)
print('Saved metadata to:', meta_path)

In [ ]:
# Example scoring function for integration into a web app or batch job
loaded_model = joblib.load(model_path)
loaded_meta = joblib.load(meta_path)

def score_new_orders(new_orders_df: pd.DataFrame):
    aligned = new_orders_df.copy()

    # Add missing columns expected by the model
    for col in loaded_meta['feature_columns']:
        if col not in aligned.columns:
            aligned[col] = np.nan

    aligned = aligned[loaded_meta['feature_columns']]
    probs = loaded_model.predict_proba(aligned)[:, 1]
    preds = (probs >= loaded_meta['optimal_threshold']).astype(int)

    result = new_orders_df.copy()
    result['fraud_probability'] = probs
    result['fraud_prediction'] = preds
    return result.sort_values('fraud_probability', ascending=False)

# Example: score the test set rows
scored_preview = score_new_orders(X_test.head(10))
display(scored_preview.head())

## 7. Final Conclusion

This notebook built a complete fraud-detection pipeline using CRISP-DM:
- defined the business problem
- explored and prepared data from SQLite
- engineered features and automated preprocessing
- trained and compared multiple classifiers
- tuned a stronger production candidate
- selected a threshold for operational use
- serialized the final model for deployment

This output can be integrated into the Chapter 17 web app so the **Run Scoring** action refreshes a fraud or late-risk priority queue.